# 1. Data Ingestion
Raw CSV load karna, basic checks, aur `Ram` / `Weight` ko numbers mein convert karna.

**Input:** `data/raw/Raw_laptop_data.csv`  
**Output:** `data/processed/01_ingested.csv`

In [1]:
import pandas as pd
import numpy as np

from common import RAW_FILE, INGESTED_FILE

In [2]:
data = pd.read_csv(RAW_FILE, encoding="utf-8-sig")
df = data.copy()
print(df.shape)
df.head()

(1303, 12)


,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [3]:
# The first column is just the old index ("Unnamed: 0"), not a feature
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
df.columns.tolist()

['Company',
 'TypeName',
 'Inches',
 'ScreenResolution',
 'Cpu',
 'Ram',
 'Memory',
 'Gpu',
 'OpSys',
 'Weight',
 'Price']

In [4]:
# Basic quality checks (duplicates must be checked AFTER dropping the index column,
# otherwise every row looks unique)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
df.info()

Missing values: 0
Duplicate rows: 29
<class 'pandas.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   str    
 1   TypeName          1303 non-null   str    
 2   Inches            1303 non-null   float64
 3   ScreenResolution  1303 non-null   str    
 4   Cpu               1303 non-null   str    
 5   Ram               1303 non-null   str    
 6   Memory            1303 non-null   str    
 7   Gpu               1303 non-null   str    
 8   OpSys             1303 non-null   str    
 9   Weight            1303 non-null   str    
 10  Price             1303 non-null   float64
dtypes: float64(2), str(9)
memory usage: 250.8 KB


In [5]:
# Ram: "8GB" -> 8
df["Ram"] = df["Ram"].str.replace("GB", "", regex=False).astype("int32")

# Weight: "1.37kg" -> 1.37
df["Weight"] = df["Weight"].str.replace("kg", "", regex=False).astype("float64")

df[["Ram", "Weight"]].describe()

,Ram,Weight
count,1303.000000,1303.000000
mean,8.382195,2.038734
std,5.084665,0.665475
min,2.000000,0.690000
25%,4.000000,1.500000
50%,8.000000,2.040000
75%,8.000000,2.300000
max,64.000000,4.700000


In [6]:
# Exact duplicates would end up in both train and test and inflate the scores -> remove
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")

# Safety checks before saving
assert df["Price"].gt(0).all(), "Price must be positive"
assert df.isna().sum().sum() == 0, "Unexpected missing values"
df.shape

Removed 29 duplicate rows


(1274, 11)

In [7]:
df.to_csv(INGESTED_FILE, index=False)
print("Saved:", INGESTED_FILE.name, df.shape)

Saved: 01_ingested.csv (1274, 11)
